# Graph Neural Network for Spatial Cascading Disaster Prediction**Goal:** Build a county-level spatial graph where each node is a US county, edgesrepresent geographic adjacency, and a GAT (Graph Attention Network) learns topredict which secondary disaster types will cascade — propagating informationacross neighboring counties via message passing.| Component | Detail ||---|---|| **Graph** | ~3,200 county nodes, ~18K adjacency edges (US Census) || **Node features** | Monthly aggregated event statistics per county || **Labels** | Multi-hot: which secondary disaster types occur (same as NB04) || **Model** | 2-layer GAT with MLP classifier head || **Visualization** | Choropleth cascade probability maps + animated propagation |

In [ ]:
import numpy as npimport pandas as pdimport pickleimport timeimport warningsimport astimport osfrom pathlib import Pathfrom collections import defaultdictimport torchimport torch.nn as nnimport torch.nn.functional as Ffrom sklearn.preprocessing import StandardScalerfrom sklearn.metrics import (    f1_score, precision_score, recall_score,    average_precision_score, hamming_loss,    precision_recall_curve, roc_curve, auc)import matplotlib.pyplot as pltimport matplotlib.ticker as mtickerimport matplotlib.colors as mcolorsfrom matplotlib.colors import LinearSegmentedColormapimport seaborn as snssns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)plt.rcParams['figure.dpi'] = 120warnings.filterwarnings('ignore')DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f"PyTorch {torch.__version__}  •  Device: {DEVICE}")%matplotlib inline

In [ ]:
# Install PyG and geo-spatial dependencies (run once)# On AWS Deep Learning AMI with CUDA, use the appropriate torch version!pip install torch-geometric geopandas contextily shapely requests --quiet

In [ ]:
import torch_geometricfrom torch_geometric.data import Data, Batchfrom torch_geometric.nn import GATConv, global_mean_poolfrom torch_geometric.utils import add_self_loops, degreeimport geopandas as gpdimport requestsprint(f"PyG {torch_geometric.__version__}")

---## 1. Data Loading & County Graph Construction

In [ ]:
DATA_DIR = Path('../data')PROCESSED_DIR = DATA_DIR / 'processed'CACHE_DIR = DATA_DIR / 'gnn_cache'CACHE_DIR.mkdir(parents=True, exist_ok=True)MODEL_DIR = Path('../models/gnn')MODEL_DIR.mkdir(parents=True, exist_ok=True)# ── Load events ──────────────────────────────────────────────────────print("Loading events_labeled_full.csv ...")events = pd.read_csv(    PROCESSED_DIR / 'events_labeled_full.csv',    usecols=[        'EVENT_ID', 'EPISODE_ID', 'EVENT_TYPE', 'STATE', 'STATE_FIPS',        'CZ_FIPS', 'CZ_TYPE', 'CZ_NAME', 'LOCATION_KEY',        'BEGIN_DATETIME', 'END_DATETIME',        'BEGIN_LAT', 'BEGIN_LON',        'MAGNITUDE', 'INJURIES_DIRECT', 'INJURIES_INDIRECT',        'DEATHS_DIRECT', 'DEATHS_INDIRECT',        'DAMAGE_PROPERTY_USD', 'DAMAGE_CROPS_USD', 'TOTAL_DAMAGE_USD',        'target', 'is_cascade',    ],    parse_dates=['BEGIN_DATETIME', 'END_DATETIME'],)print(f"  Total events: {len(events):,}")# Filter to county-type events (CZ_TYPE = 'C') — reliable FIPS mappingevents = events[events['CZ_TYPE'] == 'C'].copy()print(f"  County-type events: {len(events):,}")# Build a proper 5-digit FIPS code:  STATE_FIPS (2-digit) + CZ_FIPS (3-digit)events['FIPS'] = events['STATE_FIPS'].astype(str).str.zfill(2) + \                 events['CZ_FIPS'].astype(str).str.zfill(3)print(f"  Unique counties (FIPS): {events['FIPS'].nunique()}")# Extract year-month for temporal windowingevents['year_month'] = events['BEGIN_DATETIME'].dt.to_period('M')events.head(3)

### 1a. County Adjacency Graph (US Census Bureau)We use the [Census county adjacency file](https://www.census.gov/geographies/reference-files/2020/geo/county-adjacency.html) to build geographic neighbor relationships.

In [ ]:
# ── Download or load cached county adjacency ─────────────────────────ADJ_CACHE = CACHE_DIR / 'county_adjacency.csv'if ADJ_CACHE.exists():    print("Loading cached county adjacency ...")    adj_df = pd.read_csv(ADJ_CACHE, dtype=str)else:    print("Downloading US Census county adjacency file (2023) ...")    url = "https://www2.census.gov/geo/docs/reference/county_adjacency/county_adjacency2023.txt"    resp = requests.get(url, timeout=60)    resp.raise_for_status()    # Parse pipe-delimited file: County Name|County GEOID|Neighbor Name|Neighbor GEOID    import io    raw_df = pd.read_csv(io.StringIO(resp.text), sep='|', dtype=str)    raw_df.columns = ['county_name', 'county_fips', 'neighbor_name', 'neighbor_fips']    # Keep only FIPS columns    adj_df = raw_df[['county_fips', 'neighbor_fips']].dropna().copy()    adj_df.to_csv(ADJ_CACHE, index=False)    print(f"  Saved {len(adj_df)} adjacency records to {ADJ_CACHE}")# Remove self-adjacencyadj_df = adj_df[adj_df['county_fips'] != adj_df['neighbor_fips']].copy()print(f"Adjacency pairs (excl. self): {len(adj_df):,}")adj_df.head()

In [ ]:
# ── Build the graph ──────────────────────────────────────────────────# Nodes: counties that appear in BOTH our event data AND the adjacency fileevent_fips = set(events['FIPS'].unique())adj_fips = set(adj_df['county_fips'].unique()) | set(adj_df['neighbor_fips'].unique())common_fips = sorted(event_fips & adj_fips)print(f"Counties in events: {len(event_fips)}")print(f"Counties in adjacency: {len(adj_fips)}")print(f"Intersection (graph nodes): {len(common_fips)}")# Create FIPS → node index mappingfips_to_idx = {f: i for i, f in enumerate(common_fips)}idx_to_fips = {i: f for f, i in fips_to_idx.items()}# Filter events to only counties in the graphevents = events[events['FIPS'].isin(common_fips)].copy()print(f"Events after filtering to graph counties: {len(events):,}")# Build edge_index from adjacencyedges_src, edges_dst = [], []for _, row in adj_df.iterrows():    s, d = row['county_fips'], row['neighbor_fips']    if s in fips_to_idx and d in fips_to_idx:        edges_src.append(fips_to_idx[s])        edges_dst.append(fips_to_idx[d])edge_index = torch.tensor([edges_src, edges_dst], dtype=torch.long)# Also add reverse edges to make it undirected (adjacency is symmetric)edge_index_rev = edge_index.flip(0)edge_index = torch.cat([edge_index, edge_index_rev], dim=1)# Remove duplicatesedge_index = torch.unique(edge_index, dim=1)# Add self-loopsedge_index, _ = add_self_loops(edge_index, num_nodes=len(common_fips))N_NODES = len(common_fips)N_EDGES = edge_index.shape[1]print(f"\nGraph: {N_NODES} nodes, {N_EDGES} edges (incl. self-loops)")print(f"Avg degree: {N_EDGES / N_NODES:.1f}")

In [ ]:
# ── Compute county centroids from event lat/lon ──────────────────────county_coords = events.groupby('FIPS').agg(    lat=('BEGIN_LAT', 'median'),    lon=('BEGIN_LON', 'median'),    state=('STATE', 'first'),    name=('CZ_NAME', 'first'),).reindex(common_fips)# Fill NaN coords with 0 (a few counties may lack lat/lon)n_missing = county_coords['lat'].isna().sum()if n_missing > 0:    print(f"⚠ {n_missing} counties missing coordinates — filling with (0,0)")    county_coords = county_coords.fillna(0)print(f"County centroids computed for {len(county_coords)} counties")county_coords.head()

---## 2. Node Feature Engineering (Monthly Snapshots)Each **node** in a temporal snapshot = one county in one month.Features are aggregated statistics of all events in that county-month.

In [ ]:
# ── Define target labels (same as NB04, filtered by prevalence) ──────# Parse target column: string repr of list → actual listevents['target_parsed'] = events['target'].apply(    lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith('[') else [])# Collect all secondary disaster typesall_targets = set()for t_list in events['target_parsed']:    all_targets.update(t_list)# Filter by prevalence (>= 0.05% of county-events)target_counts = defaultdict(int)for t_list in events['target_parsed']:    for t in t_list:        target_counts[t] += 1MIN_POS_FRAC = 0.0005min_count = int(MIN_POS_FRAC * len(events))TARGET_NAMES = sorted([t for t, c in target_counts.items() if c >= min_count])N_LABELS = len(TARGET_NAMES)target_to_idx = {t: i for i, t in enumerate(TARGET_NAMES)}print(f"Target labels: {N_LABELS}")for t in TARGET_NAMES:    print(f"  {t}: {target_counts[t]:,} events ({target_counts[t]/len(events)*100:.2f}%)")

In [ ]:
# ── Get unique event types for one-hot encoding ──────────────────────EVENT_TYPES = sorted(events['EVENT_TYPE'].unique())N_EVENT_TYPES = len(EVENT_TYPES)etype_to_idx = {e: i for i, e in enumerate(EVENT_TYPES)}print(f"Event types: {N_EVENT_TYPES}")# ── Build monthly snapshots ──────────────────────────────────────────print("\nBuilding monthly county snapshots ...")time_windows = sorted(events['year_month'].unique())print(f"Time windows: {len(time_windows)} months ({time_windows[0]} to {time_windows[-1]})")def build_snapshot_features(month_events, fips_list):    """Build node feature matrix and label matrix for one month."""    n_nodes = len(fips_list)    # Feature vector per county:    # [n_events, cascade_count, cascade_rate,    #  total_damage, total_injuries, total_deaths, mean_magnitude,    #  event_type_counts (N_EVENT_TYPES)]    N_STATIC = 7    n_features = N_STATIC + N_EVENT_TYPES    X = np.zeros((n_nodes, n_features), dtype=np.float32)    Y = np.zeros((n_nodes, N_LABELS), dtype=np.float32)    for fips, grp in month_events.groupby('FIPS'):        if fips not in fips_to_idx:            continue        idx = fips_to_idx[fips]        # Aggregate features        X[idx, 0] = len(grp)                                        # n_events        X[idx, 1] = grp['is_cascade'].sum()                         # cascade_count        X[idx, 2] = grp['is_cascade'].mean()                        # cascade_rate        X[idx, 3] = np.log1p(grp['TOTAL_DAMAGE_USD'].fillna(0).sum())  # log total damage        X[idx, 4] = grp['INJURIES_DIRECT'].fillna(0).sum() + \                     grp['INJURIES_INDIRECT'].fillna(0).sum()        # total injuries        X[idx, 5] = grp['DEATHS_DIRECT'].fillna(0).sum() + \                     grp['DEATHS_INDIRECT'].fillna(0).sum()          # total deaths        X[idx, 6] = grp['MAGNITUDE'].fillna(0).mean()               # mean magnitude        # Event type counts        for _, row in grp.iterrows():            if row['EVENT_TYPE'] in etype_to_idx:                X[idx, N_STATIC + etype_to_idx[row['EVENT_TYPE']]] += 1        # Labels: which secondary disaster types cascaded in this county-month        for _, row in grp.iterrows():            for t in row['target_parsed']:                if t in target_to_idx:                    Y[idx, target_to_idx[t]] = 1.0    return X, Y# Build all snapshotssnapshots = []for ym in time_windows:    month_ev = events[events['year_month'] == ym]    X_m, Y_m = build_snapshot_features(month_ev, common_fips)    snapshots.append({        'year_month': ym,        'X': X_m,        'Y': Y_m,        'n_events': len(month_ev),    })print(f"\nBuilt {len(snapshots)} monthly snapshots")print(f"Feature dim per node: {snapshots[0]['X'].shape[1]}")print(f"Label dim per node: {snapshots[0]['Y'].shape[1]}")# Quick statstotal_events = sum(s['n_events'] for s in snapshots)avg_events = total_events / len(snapshots)print(f"Avg events/month: {avg_events:.0f}")

In [ ]:
# ── Chronological split ──────────────────────────────────────────────# Train: 2011–2022, Val: 2023, Test: 2024–2025train_snapshots = [s for s in snapshots if s['year_month'].year <= 2022]val_snapshots   = [s for s in snapshots if s['year_month'].year == 2023]test_snapshots  = [s for s in snapshots if s['year_month'].year >= 2024]print(f"Train: {len(train_snapshots)} months (2011–2022)")print(f"Val:   {len(val_snapshots)} months (2023)")print(f"Test:  {len(test_snapshots)} months (2024–2025)")# ── Normalize features using training data ───────────────────────────# Stack all training features to fit scalerX_train_all = np.concatenate([s['X'] for s in train_snapshots], axis=0)scaler = StandardScaler()scaler.fit(X_train_all)# Apply scaler to all snapshotsfor s in snapshots:    s['X_scaled'] = scaler.transform(s['X']).astype(np.float32)print(f"Feature normalization fitted on {X_train_all.shape[0]:,} county-months")# ── Compute pos_weight for class imbalance ───────────────────────────Y_train_all = np.concatenate([s['Y'] for s in train_snapshots], axis=0)pos_counts = Y_train_all.sum(axis=0)neg_counts = Y_train_all.shape[0] - pos_countspos_weight = np.clip(neg_counts / (pos_counts + 1), 1.0, 50.0)pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32).to(DEVICE)print("\nClass balance (train):")for i, t in enumerate(TARGET_NAMES):    print(f"  {t:30s}: pos={int(pos_counts[i]):6,}  neg={int(neg_counts[i]):8,}  weight={pos_weight[i]:.1f}")

---## 3. GNN Model — `SpatialCascadeGAT`A 2-layer Graph Attention Network with:- **Layer 1**: 4-head GAT → hidden_dim- **Layer 2**: 4-head GAT → hidden_dim (concat=False, so output = hidden_dim)- **MLP head**: hidden → 64 → n_labelsSpatial message passing allows each county to incorporatedisaster information from its geographic neighbors.

In [ ]:
class SpatialCascadeGAT(nn.Module):    """Graph Attention Network for spatial cascading disaster prediction.    Each node = one county, edges = geographic adjacency.    The GAT learns which neighboring counties' disaster patterns    are most informative for predicting local cascades.    """    def __init__(self, in_dim, hidden_dim=64, n_labels=16,                 n_heads=4, dropout=0.3):        super().__init__()        self.in_dim = in_dim        self.hidden_dim = hidden_dim        # Input projection        self.input_proj = nn.Sequential(            nn.Linear(in_dim, hidden_dim),            nn.LayerNorm(hidden_dim),            nn.GELU(),        )        # GAT Layer 1: multi-head attention        self.gat1 = GATConv(            hidden_dim, hidden_dim // n_heads,            heads=n_heads, dropout=dropout, concat=True,        )        self.norm1 = nn.LayerNorm(hidden_dim)        # GAT Layer 2        self.gat2 = GATConv(            hidden_dim, hidden_dim,            heads=1, dropout=dropout, concat=False,        )        self.norm2 = nn.LayerNorm(hidden_dim)        # MLP classifier head        self.classifier = nn.Sequential(            nn.Linear(hidden_dim, 128),            nn.BatchNorm1d(128),            nn.GELU(),            nn.Dropout(dropout),            nn.Linear(128, 64),            nn.BatchNorm1d(64),            nn.GELU(),            nn.Dropout(dropout * 0.5),            nn.Linear(64, n_labels),        )        self.dropout = nn.Dropout(dropout)    def forward(self, x, edge_index, return_attention=False):        # Input projection        h = self.input_proj(x)        # GAT Layer 1        h1, attn1 = self.gat1(h, edge_index, return_attention_weights=True)        h = self.norm1(F.elu(h1) + h)  # residual        h = self.dropout(h)        # GAT Layer 2        h2, attn2 = self.gat2(h, edge_index, return_attention_weights=True)        h = self.norm2(F.elu(h2) + h)  # residual        h = self.dropout(h)        # Classify each node        logits = self.classifier(h)        if return_attention:            return logits, (attn1, attn2)        return logits# Quick test_test_model = SpatialCascadeGAT(    in_dim=snapshots[0]['X'].shape[1],    n_labels=N_LABELS,)n_params = sum(p.numel() for p in _test_model.parameters())print(f"SpatialCascadeGAT defined: {n_params:,} parameters")del _test_model

---## 4. Training & Evaluation

In [ ]:
# ── Training utilities ───────────────────────────────────────────────def train_one_epoch(model, train_snaps, edge_index, criterion, optimizer):    """Train on all monthly snapshots in one epoch."""    model.train()    total_loss = 0    n_samples = 0    for snap in train_snaps:        x = torch.tensor(snap['X_scaled'], dtype=torch.float32).to(DEVICE)        y = torch.tensor(snap['Y'], dtype=torch.float32).to(DEVICE)        ei = edge_index.to(DEVICE)        optimizer.zero_grad()        logits = model(x, ei)        loss = criterion(logits, y)        loss.backward()        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)        optimizer.step()        total_loss += loss.item() * x.size(0)        n_samples += x.size(0)    return total_loss / n_samples@torch.no_grad()def evaluate(model, snaps, edge_index, criterion):    """Evaluate on a set of snapshots. Returns loss, all_probs, all_labels."""    model.eval()    total_loss = 0    n_samples = 0    all_probs, all_labels = [], []    for snap in snaps:        x = torch.tensor(snap['X_scaled'], dtype=torch.float32).to(DEVICE)        y = torch.tensor(snap['Y'], dtype=torch.float32).to(DEVICE)        ei = edge_index.to(DEVICE)        logits = model(x, ei)        loss = criterion(logits, y)        total_loss += loss.item() * x.size(0)        n_samples += x.size(0)        probs = torch.sigmoid(logits).cpu().numpy()        all_probs.append(probs)        all_labels.append(snap['Y'])    avg_loss = total_loss / n_samples    all_probs = np.concatenate(all_probs, axis=0)    all_labels = np.concatenate(all_labels, axis=0)    return avg_loss, all_probs, all_labelsdef tune_thresholds(y_true, y_prob, min_thresh=0.05):    """Per-label threshold tuning (maximise F1)."""    n_labels = y_true.shape[1]    thresholds = np.full(n_labels, 0.5)    for i in range(n_labels):        if y_true[:, i].sum() == 0:            continue        precs, recs, ths = precision_recall_curve(y_true[:, i], y_prob[:, i])        f1s = 2 * precs * recs / (precs + recs + 1e-8)        best = np.argmax(f1s)        if best < len(ths):            thresholds[i] = max(ths[best], min_thresh)    return thresholdsdef compute_metrics(y_true, y_prob, thresholds, target_names):    """Compute comprehensive metrics."""    y_pred = (y_prob >= thresholds).astype(int)    overall = {        'f1_weighted':        f1_score(y_true, y_pred, average='weighted', zero_division=0),        'f1_macro':           f1_score(y_true, y_pred, average='macro', zero_division=0),        'f1_micro':           f1_score(y_true, y_pred, average='micro', zero_division=0),        'precision_weighted': precision_score(y_true, y_pred, average='weighted', zero_division=0),        'recall_weighted':    recall_score(y_true, y_pred, average='weighted', zero_division=0),        'hamming_loss':       hamming_loss(y_true, y_pred),        'subset_accuracy':    (y_true == y_pred).all(axis=1).mean(),    }    try:        overall['aucpr_macro'] = average_precision_score(y_true, y_prob, average='macro')    except:        overall['aucpr_macro'] = 0.0    try:        per_label_roc = []        for j in range(y_true.shape[1]):            if y_true[:, j].sum() > 0 and y_true[:, j].sum() < len(y_true):                fpr, tpr, _ = roc_curve(y_true[:, j], y_prob[:, j])                per_label_roc.append(auc(fpr, tpr))        overall['roc_auc_macro'] = np.mean(per_label_roc) if per_label_roc else 0.0    except:        overall['roc_auc_macro'] = 0.0    per_label = []    for j, lbl in enumerate(target_names):        sup = int(y_true[:, j].sum())        f1  = f1_score(y_true[:, j], y_pred[:, j], zero_division=0)        prec = precision_score(y_true[:, j], y_pred[:, j], zero_division=0)        rec  = recall_score(y_true[:, j], y_pred[:, j], zero_division=0)        try:            prauc = average_precision_score(y_true[:, j], y_prob[:, j])        except:            prauc = 0.0        per_label.append(dict(label=lbl, support=sup, f1=f1, precision=prec,                              recall=rec, pr_auc=prauc))    return overall, pd.DataFrame(per_label)print("Training utilities defined.")

In [ ]:
# ── Train the GNN ────────────────────────────────────────────────────N_FEATURES = snapshots[0]['X_scaled'].shape[1]EPOCHS = 80PATIENCE = 15LR = 1e-3WEIGHT_DECAY = 1e-4HIDDEN_DIM = 64model = SpatialCascadeGAT(    in_dim=N_FEATURES,    hidden_dim=HIDDEN_DIM,    n_labels=N_LABELS,    n_heads=4,    dropout=0.3,).to(DEVICE)optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(    optimizer, mode='max', factor=0.5, patience=5)criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)print(f"Model: SpatialCascadeGAT")print(f"  Features: {N_FEATURES}, Hidden: {HIDDEN_DIM}, Labels: {N_LABELS}")print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")print(f"  Epochs: {EPOCHS}, Patience: {PATIENCE}, LR: {LR}")print(f"\nTraining on {len(train_snapshots)} monthly snapshots ...")print("=" * 70)best_val_f1 = -1best_state = Nonewait = 0history = {'train_loss': [], 'val_loss': [], 'val_f1': []}t_start = time.time()for epoch in range(1, EPOCHS + 1):    # Train    train_loss = train_one_epoch(model, train_snapshots, edge_index, criterion, optimizer)    # Validate    val_loss, val_probs, val_labels = evaluate(model, val_snapshots, edge_index, criterion)    val_preds = (val_probs >= 0.5).astype(int)    val_f1 = f1_score(val_labels, val_preds, average='macro', zero_division=0)    scheduler.step(val_f1)    history['train_loss'].append(train_loss)    history['val_loss'].append(val_loss)    history['val_f1'].append(val_f1)    if val_f1 > best_val_f1:        best_val_f1 = val_f1        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}        wait = 0    else:        wait += 1    if epoch % 5 == 0 or epoch == 1 or wait == 0:        marker = ' ★' if wait == 0 else ''        elapsed = time.time() - t_start        print(f"  Epoch {epoch:3d}  loss={train_loss:.4f}  val_loss={val_loss:.4f}  "              f"val_f1={val_f1:.4f}  [{elapsed:.0f}s]{marker}")    if wait >= PATIENCE:        print(f"  Early stop at epoch {epoch} (best val_f1={best_val_f1:.4f})")        break# Restore best modelmodel.load_state_dict(best_state)model.to(DEVICE)elapsed = time.time() - t_startprint(f"\nTraining complete in {elapsed:.0f}s. Best val F1 (macro): {best_val_f1:.4f}")

In [ ]:
# ── Threshold tuning on validation set ───────────────────────────────val_loss_final, val_probs, val_labels = evaluate(model, val_snapshots, edge_index, criterion)thresholds = tune_thresholds(val_labels, val_probs)print("Tuned per-label thresholds:")for i, t in enumerate(TARGET_NAMES):    print(f"  {t:30s}: {thresholds[i]:.3f}")

In [ ]:
# ── Test-set evaluation ──────────────────────────────────────────────test_loss, test_probs, test_labels = evaluate(model, test_snapshots, edge_index, criterion)overall_metrics, per_label_df = compute_metrics(    test_labels, test_probs, thresholds, TARGET_NAMES)print("=" * 70)print("TEST SET RESULTS — SpatialCascadeGAT")print("=" * 70)for k, v in overall_metrics.items():    print(f"  {k:25s}: {v:.4f}")print("\nPer-label breakdown:")display(per_label_df.sort_values('f1', ascending=False).reset_index(drop=True).style.format({    'f1': '{:.3f}', 'precision': '{:.3f}', 'recall': '{:.3f}', 'pr_auc': '{:.3f}',}).background_gradient(subset=['f1'], cmap='RdYlGn'))

In [ ]:
# ── Save model and results ───────────────────────────────────────────torch.save(model.state_dict(), MODEL_DIR / 'spatial_cascade_gat.pt')with open(MODEL_DIR / 'gat_results.pkl', 'wb') as f:    pickle.dump({        'overall': overall_metrics,        'per_label': per_label_df.to_dict(),        'history': history,        'thresholds': thresholds.tolist(),        'fips_to_idx': fips_to_idx,        'target_names': TARGET_NAMES,    }, f)print(f"Model saved to {MODEL_DIR / 'spatial_cascade_gat.pt'}")

In [ ]:
# ── Training curves ──────────────────────────────────────────────────fig, ax = plt.subplots(1, 1, figsize=(10, 5))epochs_range = range(1, len(history['train_loss']) + 1)l1, = ax.plot(epochs_range, history['train_loss'], 'b-', alpha=0.6, label='Train Loss')l2, = ax.plot(epochs_range, history['val_loss'], 'r-', alpha=0.6, label='Val Loss')ax.set_xlabel('Epoch')ax.set_ylabel('Loss')ax2 = ax.twinx()l3, = ax2.plot(epochs_range, history['val_f1'], 'g--', alpha=0.8, label='Val F1 (macro)')ax2.set_ylabel('F1 Score')ax.legend(handles=[l1, l2, l3], loc='center right')ax.set_title('SpatialCascadeGAT — Training Curves')plt.tight_layout()plt.savefig(str(MODEL_DIR / 'gat_training_curves.png'), dpi=150, bbox_inches='tight')plt.show()

---## 5. Spatial Cascade Propagation VisualizationThree visualizations:1. **Choropleth** — predicted cascade probability by county2. **Cascade propagation animation** — how disasters spread across counties3. **GAT attention heatmap** — which county-to-county edges the model attends to most

In [ ]:
# ── Download US county shapefile for choropleth ───────────────────────SHAPEFILE_CACHE = CACHE_DIR / 'us_counties.gpkg'if SHAPEFILE_CACHE.exists():    print("Loading cached county shapefile ...")    counties_gdf = gpd.read_file(SHAPEFILE_CACHE)else:    print("Downloading US county shapefile (Census TIGER/Line 2020) ...")    tiger_url = (        "https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_us_county_20m.zip"    )    counties_gdf = gpd.read_file(tiger_url)    counties_gdf.to_file(SHAPEFILE_CACHE, driver='GPKG')    print(f"  Saved to {SHAPEFILE_CACHE}")# Create FIPS column matching our formatcounties_gdf['FIPS'] = counties_gdf['STATEFP'] + counties_gdf['COUNTYFP']print(f"Shapefile loaded: {len(counties_gdf)} county geometries")# Filter to CONUS for cleaner visualizationEXCLUDE_STATES = {'02', '15', '60', '66', '69', '72', '78'}  # AK, HI, territoriescounties_conus = counties_gdf[~counties_gdf['STATEFP'].isin(EXCLUDE_STATES)].copy()print(f"CONUS counties: {len(counties_conus)}")

In [ ]:
# ── 5a. Choropleth: Predicted Cascade Probability ────────────────────# Use the last test month's predictionslast_test_snap = test_snapshots[-1]model.eval()with torch.no_grad():    x = torch.tensor(last_test_snap['X_scaled'], dtype=torch.float32).to(DEVICE)    ei = edge_index.to(DEVICE)    logits = model(x, ei)    probs = torch.sigmoid(logits).cpu().numpy()# Max cascade probability across all labels per countymax_cascade_prob = probs.max(axis=1)# Build a DataFrame: FIPS → probabilityprob_df = pd.DataFrame({    'FIPS': common_fips,    'max_cascade_prob': max_cascade_prob,    'any_cascade_true': last_test_snap['Y'].max(axis=1),})# Merge with shapefilemerged = counties_conus.merge(prob_df, on='FIPS', how='left')merged['max_cascade_prob'] = merged['max_cascade_prob'].fillna(0)# Plotfig, axes = plt.subplots(1, 2, figsize=(20, 8))# Predicted probabilitiesmerged.plot(    column='max_cascade_prob',    ax=axes[0],    cmap='YlOrRd',    legend=True,    edgecolor='#cccccc',    linewidth=0.1,    legend_kwds={'label': 'Max Cascade Probability', 'shrink': 0.6},    missing_kwds={'color': '#f0f0f0'},)axes[0].set_title(    f'Predicted Cascade Probability — {last_test_snap["year_month"]}',    fontsize=14, fontweight='bold')axes[0].set_xlim(-130, -65)axes[0].set_ylim(24, 50)axes[0].set_axis_off()# Ground truthtruth_df = prob_df.copy()merged_truth = counties_conus.merge(truth_df, on='FIPS', how='left')merged_truth['any_cascade_true'] = merged_truth['any_cascade_true'].fillna(0)merged_truth.plot(    column='any_cascade_true',    ax=axes[1],    cmap='YlOrRd',    legend=True,    edgecolor='#cccccc',    linewidth=0.1,    legend_kwds={'label': 'Any Cascade (Ground Truth)', 'shrink': 0.6},    missing_kwds={'color': '#f0f0f0'},)axes[1].set_title(    f'Ground Truth Cascades — {last_test_snap["year_month"]}',    fontsize=14, fontweight='bold')axes[1].set_xlim(-130, -65)axes[1].set_ylim(24, 50)axes[1].set_axis_off()plt.suptitle('SpatialCascadeGAT — County-Level Cascade Predictions vs Ground Truth',             fontsize=16, fontweight='bold', y=1.02)plt.tight_layout()plt.savefig(str(MODEL_DIR / 'cascade_choropleth.png'), dpi=150, bbox_inches='tight')plt.show()

In [ ]:
# ── 5b. Animated Cascade Propagation ─────────────────────────────────from matplotlib.animation import FuncAnimationfrom IPython.display import HTML# Select a sequence of months to animate (last 6 test months)anim_snaps = test_snapshots[-6:] if len(test_snapshots) >= 6 else test_snapshots# Pre-compute predictions for each monthanim_data = []model.eval()for snap in anim_snaps:    with torch.no_grad():        x = torch.tensor(snap['X_scaled'], dtype=torch.float32).to(DEVICE)        ei = edge_index.to(DEVICE)        logits = model(x, ei)        probs = torch.sigmoid(logits).cpu().numpy()    max_prob = probs.max(axis=1)    anim_data.append({        'year_month': str(snap['year_month']),        'max_prob': max_prob,        'n_events': snap['n_events'],    })# Create animationfig, ax = plt.subplots(1, 1, figsize=(14, 8))# Normalize colormap across all framesvmax = max(d['max_prob'].max() for d in anim_data)vmin = 0def animate(frame):    ax.clear()    d = anim_data[frame]    prob_df = pd.DataFrame({        'FIPS': common_fips,        'max_cascade_prob': d['max_prob'],    })    merged = counties_conus.merge(prob_df, on='FIPS', how='left')    merged['max_cascade_prob'] = merged['max_cascade_prob'].fillna(0)    merged.plot(        column='max_cascade_prob',        ax=ax,        cmap='YlOrRd',        vmin=vmin,        vmax=vmax,        edgecolor='#cccccc',        linewidth=0.1,        missing_kwds={'color': '#f0f0f0'},    )    ax.set_title(        f'Predicted Cascade Propagation — {d["year_month"]}\n'        f'({d["n_events"]:,} events)',        fontsize=14, fontweight='bold'    )    ax.set_xlim(-130, -65)    ax.set_ylim(24, 50)    ax.set_axis_off()anim = FuncAnimation(fig, animate, frames=len(anim_data), interval=1200, repeat=True)# Save as GIFgif_path = MODEL_DIR / 'cascade_propagation.gif'anim.save(str(gif_path), writer='pillow', fps=1)print(f"Animation saved to {gif_path}")# Display in notebookplt.close(fig)HTML(anim.to_jshtml())

In [ ]:
# ── 5c. GAT Attention Weight Visualization ───────────────────────────# Get attention weights from the last test snapshotmodel.eval()with torch.no_grad():    x = torch.tensor(last_test_snap['X_scaled'], dtype=torch.float32).to(DEVICE)    ei = edge_index.to(DEVICE)    logits, (attn1, attn2) = model(x, ei, return_attention=True)# attn1 = (edge_index_with_self_loops, attention_coefficients)attn_edge_index = attn1[0].cpu()attn_weights = attn1[1].cpu().numpy()  # shape: (n_edges, n_heads)# Average attention across headsavg_attn = attn_weights.mean(axis=1) if attn_weights.ndim > 1 else attn_weights# Find top-K highest attention edges (excluding self-loops)non_self = attn_edge_index[0] != attn_edge_index[1]non_self_idx = non_self.numpy()top_k = min(200, non_self_idx.sum())top_edges_idx = np.argsort(avg_attn[non_self_idx])[-top_k:]non_self_edges = attn_edge_index[:, non_self].numpy()fig, ax = plt.subplots(1, 1, figsize=(14, 8))# Plot county outlines as backgroundcounties_conus.plot(ax=ax, color='#f5f5f5', edgecolor='#cccccc', linewidth=0.2)# Plot attention edgesfor idx in top_edges_idx:    src_fips = idx_to_fips.get(non_self_edges[0, idx])    dst_fips = idx_to_fips.get(non_self_edges[1, idx])    if src_fips and dst_fips:        src_coord = county_coords.loc[src_fips] if src_fips in county_coords.index else None        dst_coord = county_coords.loc[dst_fips] if dst_fips in county_coords.index else None        if src_coord is not None and dst_coord is not None:            w = avg_attn[non_self_idx][idx]            ax.plot(                [src_coord['lon'], dst_coord['lon']],                [src_coord['lat'], dst_coord['lat']],                color='red', alpha=min(w * 3, 0.8), linewidth=w * 5,            )ax.set_title(    f'GAT Attention Weights — Top {top_k} County-to-County Connections\n'    f'{last_test_snap["year_month"]}',    fontsize=14, fontweight='bold')ax.set_xlim(-130, -65)ax.set_ylim(24, 50)ax.set_axis_off()plt.tight_layout()plt.savefig(str(MODEL_DIR / 'gat_attention_map.png'), dpi=150, bbox_inches='tight')plt.show()

### 5d. 🎬 GNN Message-Passing Propagation VideoA cinematic animation showing **how the GNN spreads disaster information across the county graph**:1. **Seed frame** — raw event activity lights up source counties2. **Message passing rounds** — each GAT layer propagates risk to neighbors; edges glow with attention weight intensity3. **Cascade prediction** — final predicted cascade probabilities appear as a heat-map4. **Temporal sweep** — steps through consecutive months showing how cascades evolveThe video is saved as a GIF (+ MP4 if ffmpeg is available) and displayed inline.

In [ ]:
# ── 5d. GNN Message-Passing Propagation Video ────────────────────────────
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.collections import LineCollection
from IPython.display import HTML
import matplotlib.patheffects as pe

# ---- Helper: get per-layer activations + attention ----
@torch.no_grad()
def get_layer_activations(model, x_tensor, edge_index_dev):
    """Run the GAT and capture intermediate node embeddings + attention."""
    model.eval()
    h0 = model.input_proj(x_tensor)

    h1_raw, attn1 = model.gat1(h0, edge_index_dev,
                                return_attention_weights=True)
    h1 = model.norm1(F.elu(h1_raw) + h0)

    h2_raw, attn2 = model.gat2(h1, edge_index_dev,
                                return_attention_weights=True)
    h2 = model.norm2(F.elu(h2_raw) + h1)

    logits = model.classifier(h2)
    probs  = torch.sigmoid(logits)

    stages = {
        'input':  x_tensor.cpu().numpy(),
        'proj':   h0.cpu().numpy(),
        'gat1':   h1.cpu().numpy(),
        'gat2':   h2.cpu().numpy(),
        'probs':  probs.cpu().numpy(),
    }
    norms = {k: np.linalg.norm(v, axis=1) for k,v in stages.items()}

    attn_ei_1 = attn1[0].cpu().numpy()
    attn_w_1  = attn1[1].cpu().numpy()
    if attn_w_1.ndim > 1:
        attn_w_1 = attn_w_1.mean(axis=1)

    attn_ei_2 = attn2[0].cpu().numpy()
    attn_w_2  = attn2[1].cpu().numpy()
    if attn_w_2.ndim > 1:
        attn_w_2 = attn_w_2.mean(axis=1)

    return norms, stages, (attn_ei_1, attn_w_1), (attn_ei_2, attn_w_2)


# ---- Select months for the video ----
n_video_months = min(8, len(test_snapshots))
video_snaps = test_snapshots[:n_video_months]

video_data = []
for snap in video_snaps:
    x_t = torch.tensor(snap['X_scaled'], dtype=torch.float32).to(DEVICE)
    ei_d = edge_index.to(DEVICE)
    norms, stages, a1, a2 = get_layer_activations(model, x_t, ei_d)
    video_data.append({
        'ym':    str(snap['year_month']),
        'norms': norms,
        'probs': stages['probs'],
        'attn1': a1,
        'attn2': a2,
        'y_true': snap['Y'],
    })

print(f"Pre-computed activations for {len(video_data)} months")

# ---- Coordinate arrays ----
lons = county_coords['lon'].values
lats = county_coords['lat'].values
mask_conus = (lons < -60) & (lons > -130) & (lats > 24) & (lats < 50)

# ---- Build CONUS edge segments (skip self-loops) ----
ei_np = edge_index.numpy()
edge_segments = []
edge_src_idx = []
for e in range(ei_np.shape[1]):
    s, d = ei_np[0, e], ei_np[1, e]
    if s == d:
        continue
    if not (mask_conus[s] and mask_conus[d]):
        continue
    edge_segments.append([(lons[s], lats[s]), (lons[d], lats[d])])
    edge_src_idx.append((s, d))

print(f"CONUS edge segments for viz: {len(edge_segments)}")


In [ ]:
# ── Render the propagation video ─────────────────────────────────────────
# Animation: for each month, 5 stages:
#   0: raw input   1: projection   2: GAT-1 (edges glow)
#   3: GAT-2 (edges glow)          4: predicted probabilities

STAGES = ['input', 'proj', 'gat1', 'gat2', 'probs']
STAGE_LABELS = [
    'Raw Input Activity',
    'Input Projection',
    'GAT Layer 1 — Message Passing',
    'GAT Layer 2 — Message Passing',
    'Predicted Cascade Probability',
]

frames_per_month = len(STAGES)
total_frames = len(video_data) * frames_per_month
print(f"Total frames: {total_frames} ({len(video_data)} months \u00d7 {frames_per_month} stages)")

# Color scheme
BACKGROUND = '#0a0e27'
EDGE_DORMANT = '#1a1f4e'
NODE_CMAP = LinearSegmentedColormap.from_list(
    'disaster', ['#0a0e27', '#1b3a6b', '#2980b9', '#f39c12', '#e74c3c', '#ff1744']
)
EDGE_CMAP = LinearSegmentedColormap.from_list(
    'attention', ['#0a0e2700', '#2980b944', '#f39c1288', '#e74c3ccc', '#ff1744ff']
)

fig, ax = plt.subplots(figsize=(16, 9), facecolor=BACKGROUND)
fig.subplots_adjust(left=0.02, right=0.98, top=0.92, bottom=0.02)

counties_conus.boundary.plot(ax=ax, linewidth=0.08, color='#ffffff15')
ax.set_xlim(-128, -65)
ax.set_ylim(24, 50)
ax.set_facecolor(BACKGROUND)
ax.set_axis_off()

title_text = ax.text(
    0.5, 0.97, '', transform=ax.transAxes, fontsize=16,
    color='white', fontweight='bold', ha='center', va='top',
    fontfamily='monospace',
    path_effects=[pe.withStroke(linewidth=3, foreground=BACKGROUND)]
)
subtitle_text = ax.text(
    0.5, 0.93, '', transform=ax.transAxes, fontsize=11,
    color='#8899bb', ha='center', va='top', fontfamily='monospace',
)

node_scatter = ax.scatter([], [], s=[], c=[], cmap=NODE_CMAP, zorder=5,
                          edgecolors='none', vmin=0, vmax=1)
edge_lc = LineCollection([], linewidths=0.3, colors=EDGE_DORMANT, zorder=2)
ax.add_collection(edge_lc)


def animate(frame_idx):
    month_idx = frame_idx // frames_per_month
    stage_idx = frame_idx % frames_per_month
    d = video_data[month_idx]
    stage = STAGES[stage_idx]

    # Node intensities
    if stage == 'probs':
        intensity = d['probs'].max(axis=1)
    else:
        raw = d['norms'][stage]
        lo, hi = np.percentile(raw[mask_conus], [5, 95])
        intensity = np.clip((raw - lo) / (hi - lo + 1e-8), 0, 1)

    sizes = 1 + intensity * 35
    sizes[~mask_conus] = 0

    offsets = np.column_stack([lons, lats])
    node_scatter.set_offsets(offsets)
    node_scatter.set_array(intensity)
    node_scatter.set_sizes(sizes)
    node_scatter.set_clim(0, 1)

    # Edge glow (only during GAT layers)
    if stage in ('gat1', 'gat2'):
        attn_ei, attn_w = d['attn1'] if stage == 'gat1' else d['attn2']
        attn_lookup = {}
        for e_i in range(attn_ei.shape[1]):
            s, dd = int(attn_ei[0, e_i]), int(attn_ei[1, e_i])
            if s != dd:
                attn_lookup[(s, dd)] = float(attn_w[e_i])

        edge_colors = []
        edge_widths = []
        for (s, dd) in edge_src_idx:
            w = attn_lookup.get((s, dd), 0.0)
            w_normed = min(w * 8, 1.0)
            edge_colors.append(EDGE_CMAP(w_normed))
            edge_widths.append(0.2 + w_normed * 2.5)

        edge_lc.set_segments(edge_segments)
        edge_lc.set_colors(edge_colors)
        edge_lc.set_linewidths(edge_widths)
    else:
        edge_lc.set_segments(edge_segments)
        edge_lc.set_colors([EDGE_DORMANT] * len(edge_segments))
        edge_lc.set_linewidths([0.15] * len(edge_segments))

    title_text.set_text(f'GNN Cascade Propagation  \u2014  {d["ym"]}')
    subtitle_text.set_text(f'Stage {stage_idx+1}/{frames_per_month}: {STAGE_LABELS[stage_idx]}')
    return node_scatter, edge_lc, title_text, subtitle_text


anim = FuncAnimation(fig, animate, frames=total_frames,
                     interval=800, blit=False, repeat=True)

video_path = MODEL_DIR / 'gnn_propagation_video.gif'
anim.save(str(video_path), writer=PillowWriter(fps=2), dpi=120,
          savefig_kwargs={'facecolor': BACKGROUND})
print(f"\u2705 Saved propagation video: {video_path}")

# Try MP4 if ffmpeg available
try:
    from matplotlib.animation import FFMpegWriter
    mp4_path = MODEL_DIR / 'gnn_propagation_video.mp4'
    writer = FFMpegWriter(fps=2, metadata={'title': 'GNN Cascade Propagation'})
    anim.save(str(mp4_path), writer=writer, dpi=120,
              savefig_kwargs={'facecolor': BACKGROUND})
    print(f"\u2705 Saved MP4: {mp4_path}")
except Exception as e:
    print(f"\u26a0 MP4 not saved (install ffmpeg): {e}")

plt.close(fig)
HTML(anim.to_jshtml())


In [ ]:
# ── 5e. Single-Episode Deep Dive — Watch One Disaster Cascade ────────
# Find month with most cascade activity, zoom into the epicentre

cascade_rates = [(i, d['y_true'].max(axis=1).mean()) for i,d in enumerate(video_data)]
best_month_idx = max(cascade_rates, key=lambda x: x[1])[0]
d = video_data[best_month_idx]
ym_str = d['ym']
cr = cascade_rates[best_month_idx][1]
print(f'Highest-cascade month: {ym_str} (cascade rate: {cr:.3f})')

# Find the county with highest predicted cascade probability
probs_max = d['probs'].max(axis=1)
epicenter_idx = np.argmax(probs_max * mask_conus)
epi_lon, epi_lat = lons[epicenter_idx], lats[epicenter_idx]
epi_fips = idx_to_fips[epicenter_idx]
epi_name = county_coords.loc[epi_fips, 'name'] if epi_fips in county_coords.index else epi_fips
epi_state = county_coords.loc[epi_fips, 'state'] if epi_fips in county_coords.index else ''
mc = probs_max[epicenter_idx]
print(f'Epicenter: {epi_name}, {epi_state} (FIPS {epi_fips}) \u2014 max cascade prob: {mc:.3f}')

# Zoom window: \u00b14 degrees around epicenter
ZOOM = 4
zoom_box = (epi_lon - ZOOM, epi_lon + ZOOM, epi_lat - ZOOM/1.5, epi_lat + ZOOM/1.5)

fig2, ax2 = plt.subplots(figsize=(14, 9), facecolor=BACKGROUND)
fig2.subplots_adjust(left=0.02, right=0.98, top=0.90, bottom=0.02)

counties_conus.boundary.plot(ax=ax2, linewidth=0.3, color='#ffffff22')
ax2.set_xlim(zoom_box[0], zoom_box[1])
ax2.set_ylim(zoom_box[2], zoom_box[3])
ax2.set_facecolor(BACKGROUND)
ax2.set_axis_off()

title2 = ax2.text(0.5, 0.97, '', transform=ax2.transAxes, fontsize=15,
                  color='white', fontweight='bold', ha='center', va='top',
                  fontfamily='monospace',
                  path_effects=[pe.withStroke(linewidth=3, foreground=BACKGROUND)])
sub2 = ax2.text(0.5, 0.93, '', transform=ax2.transAxes, fontsize=11,
                color='#8899bb', ha='center', va='top', fontfamily='monospace')

# Mark epicenter with a star
ax2.plot(epi_lon, epi_lat, marker='*', markersize=18, color='#ff1744',
         zorder=10, markeredgecolor='white', markeredgewidth=0.8)

zoom_mask = ((lons > zoom_box[0]) & (lons < zoom_box[1]) &
             (lats > zoom_box[2]) & (lats < zoom_box[3]))

# Zoomed edge segments
zoom_edge_segs = []
zoom_edge_pairs = []
for seg, (s, dd) in zip(edge_segments, edge_src_idx):
    if zoom_mask[s] or zoom_mask[dd]:
        zoom_edge_segs.append(seg)
        zoom_edge_pairs.append((s, dd))

scatter2 = ax2.scatter([], [], s=[], c=[], cmap=NODE_CMAP, zorder=5,
                       edgecolors='none', vmin=0, vmax=1)
lc2 = LineCollection([], linewidths=0.5, colors=EDGE_DORMANT, zorder=2)
ax2.add_collection(lc2)


def animate_zoom(stage_idx):
    stage = STAGES[stage_idx]

    if stage == 'probs':
        intensity = d['probs'].max(axis=1)
    else:
        raw = d['norms'][stage]
        lo, hi = np.percentile(raw[zoom_mask], [5, 95]) if zoom_mask.sum() > 0 else (0, 1)
        intensity = np.clip((raw - lo) / (hi - lo + 1e-8), 0, 1)

    sizes = np.where(zoom_mask, 8 + intensity * 120, 0)
    offsets = np.column_stack([lons, lats])
    scatter2.set_offsets(offsets)
    scatter2.set_array(intensity)
    scatter2.set_sizes(sizes)
    scatter2.set_clim(0, 1)

    if stage in ('gat1', 'gat2'):
        attn_ei, attn_w = d['attn1'] if stage == 'gat1' else d['attn2']
        attn_lookup = {}
        for e_i in range(attn_ei.shape[1]):
            s, dd2 = int(attn_ei[0, e_i]), int(attn_ei[1, e_i])
            if s != dd2:
                attn_lookup[(s, dd2)] = float(attn_w[e_i])

        ec, ew = [], []
        for (s, dd2) in zoom_edge_pairs:
            w = attn_lookup.get((s, dd2), 0.0)
            w_n = min(w * 8, 1.0)
            ec.append(EDGE_CMAP(w_n))
            ew.append(0.5 + w_n * 4)
        lc2.set_segments(zoom_edge_segs)
        lc2.set_colors(ec)
        lc2.set_linewidths(ew)
    else:
        lc2.set_segments(zoom_edge_segs)
        lc2.set_colors([EDGE_DORMANT] * len(zoom_edge_segs))
        lc2.set_linewidths([0.3] * len(zoom_edge_segs))

    title2.set_text(f'Cascade Deep Dive \u2014 {epi_name}, {epi_state}  [{d["ym"]}]')
    sub2.set_text(f'Stage {stage_idx+1}/{len(STAGES)}: {STAGE_LABELS[stage_idx]}')
    return scatter2, lc2, title2, sub2


anim2 = FuncAnimation(fig2, animate_zoom, frames=len(STAGES),
                       interval=1500, blit=False, repeat=True)

zoom_path = MODEL_DIR / 'gnn_cascade_deepdive.gif'
anim2.save(str(zoom_path), writer=PillowWriter(fps=1), dpi=130,
           savefig_kwargs={'facecolor': BACKGROUND})
print(f"\u2705 Saved deep-dive: {zoom_path}")

plt.close(fig2)
HTML(anim2.to_jshtml())


---## 6. Summary| Metric | SpatialCascadeGAT ||---|---|| F1 (macro) | See results above || F1 (weighted) | See results above || AUCPR (macro) | See results above || ROC-AUC (macro) | See results above |**Key findings:**- The GAT architecture allows each county to incorporate disaster patterns from geographic neighbors- Attention weights reveal which county-to-county connections are most predictive of cascading disasters- The choropleth visualization shows spatial clustering of cascade risk- Temporal animation reveals how disaster propagation patterns evolve month-to-month